# 1. vllm 라이브러리 설치

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.10.2 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
!uv pip install --system vllm==0.11.0 torch==2.8.0 transformers==4.57.6

Using Python 3.12.12 environment at: /usr
Resolved 163 packages in 490ms
Prepared 9 packages in 27.18s
Uninstalled 9 packages in 264ms
Installed 9 packages in 312ms
 - anthropic==0.79.0
 + anthropic==0.71.0
 - compressed-tensors==0.13.0
 + compressed-tensors==0.12.2
 - flashinfer-python==0.6.1
 + flashinfer-python==0.5.3
 - torch==2.9.1
 + torch==2.9.0
 - torchaudio==2.9.1
 + torchaudio==2.9.0
 - torchvision==0.24.1
 + torchvision==0.24.0
 - triton==3.5.1
 + triton==3.5.0
 - vllm==0.15.0
 + vllm==0.13.0
 - xgrammar==0.1.29
 + xgrammar==0.1.27


# 2. 모델 로드

## 2-1. Exaone 모델 로드

In [47]:
!vllm serve LGAI-EXAONE/EXAONE-4.0-1.2B \
--max_model_len 2048 \
--gpu-memory-utilization 0.45 \
--port 8000 > exaone_server.log 2>&1 &

In [57]:
!cat exaone_server.log

2026-02-16 10:05:39.722102: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771236339.756412   14055 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771236339.771104   14055 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771236339.796815   14055 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771236339.796845   14055 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771236339.796850   14055 computation_placer.cc:177] computation placer alr

## 2-2. Qwen 모델 로드

In [67]:
!vllm serve Qwen/Qwen3-0.6B \
--max_model_len 2048 \
--gpu-memory-utilization 0.45 \
--port 8001 > qwen_server.log 2>&1 &

In [71]:
cat qwen_server.log

2026-02-16 10:10:52.676913: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771236652.705020   15942 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771236652.720452   15942 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771236652.754332   15942 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771236652.754382   15942 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771236652.754386   15942 computation_placer.cc:177] computation placer alr

## 2-3. 프로세스별 GPU 사용량 확인

In [72]:
!nvidia-smi

Mon Feb 16 10:15:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             27W /   70W |   14315MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 3. 모델 추론

In [59]:
messages = [
    {"role": "user", "content": "서울의 경복궁은 어떤 곳이야?"}
]

In [60]:
payload = {
    "messages": messages,
    "temperature": 0.7
}

headers={"Content-Type": "application/json"}

In [61]:
eaxaone_endpoint = "http://localhost:8000/v1/chat/completions"
qwen_endpoint = "http://localhost:8001/v1/chat/completions"

## 3-1. Exaone 모델 요청 및 응답

In [62]:
# Exaone 모델 요청
import httpx

async with httpx.AsyncClient(timeout=30.0) as client:
    response = await client.post(url=eaxaone_endpoint, json=payload, headers=headers)

In [63]:
# Exaone 모델 응답
import json

if response.status_code == 200:
    model_response = response.json()
    print(json.dumps(model_response, indent=4, ensure_ascii=False))

else:
    print(f"에러 발생: {response.status_code}")
    print(response.text)

{
    "id": "chatcmpl-84c6cd5ad1f0b2da",
    "object": "chat.completion",
    "created": 1771236517,
    "model": "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "서울의 경복궁은 조선 왕조의 정궁으로, 서울 종로구 thoroughhead로 located 되어 있습니다. 주요 특징은 다음과 같습니다:\n\n1. **역사적 배경**: 1395년 조선 태조 이성계에 의해 건설된 조선의ودigence palace로, 고려 후기부터 사용되다가 1572년 임진왜란 때 파�되었습니다.\n\n2. **건축 양식**: 순수 한옥 양식으로 지어졌으며,ventional 한국 궁궐의 전형을 보여줍니다. 주요 건물로는 근정전(Main Hall), 교대정(Gyeongje Fortress), 경운궁(덕소궁, 현재 덕수궁 내 housed) 등이 있습니다.\n\n3. **현재 상태**: 부분적으로 복원되어 있으며, 국립중앙박물관과 연결된 경운궁은 함께 유네스코 세계문화유산으로 등재되었습니다.\n\n4. **문화적 의미**: 조선 시대의 정치·religious 중심지였으며, 조선의 통치 이념과 문화를 상징하는 중요한 유적지입니다.",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [],
                "reasoning": null,
                "reasoning_

## 3-2. Qwen 모델 요청 및 응답

In [ ]:
# Qwen 모델

async with httpx.AsyncClient(timeout=30.0) as client:
    response = await client.post(url=qwen_endpoint, json=payload, headers=headers)

In [ ]:
# Qwen 모델 응답

if response.status_code == 200:
    model_response = response.json()
    print(json.dumps(model_response, indent=4, ensure_ascii=False))

else:
    print(f"에러 발생: {response.status_code}")
    print(response.text)

{
    "id": "chatcmpl-af3b0b704bb1a84c",
    "object": "chat.completion",
    "created": 1771236928,
    "model": "Qwen/Qwen3-0.6B",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "<think>\nOkay, the user is asking about the location of the Seoul Pyeongchang Olympic Park. I need to confirm the correct name and where it is located. First, I know that Pyeongchang is the name of the Olympic Park, and it's in South Korea. The user might be a tourist or someone interested in sports facilities. I should mention the city where it's located, which is Pyeongchang, South Korea. Also, it's important to note that it's the home of the Olympic Games and has a large sports complex. Let me make sure the information is accurate and clear.\n</think>\n\n서울의 경복궁은 **경복궁**에서 위치합니다. 경복궁은 **서울의 중심부**에 위치하며, 서울에서 가장 중요한 공공 공간으로 이용됩니다. 이곳은 올해 2028년 세계 올림픽에서 열린 **경복궁 2028 올림픽**을 대상으로 한 공공 건물입니다.",
                "refusa